# AIReceptionist Architecture Guide

This notebook provides a step-by-step walkthrough of the AIReceptionist codebase - a production-grade AI phone receptionist built on:

- **OpenAI Realtime API** - Direct speech-to-speech AI (no cascaded STT→LLM→TTS)
- **LiveKit Agents SDK** - Real-time communication framework for handling calls
- **Pydantic v2** - Type-safe configuration validation
- **YAML** - Human-readable business configuration

---

## Table of Contents

1. [Project Overview](#1-project-overview)
2. [Data Flow Architecture](#2-data-flow-architecture)
3. [Configuration System](#3-configuration-system)
4. [System Prompt Building](#4-system-prompt-building)
5. [Agent Entry Point](#5-agent-entry-point)
6. [Call Lifecycle Management](#6-call-lifecycle-management)
7. [Tool Implementations](#7-tool-implementations)
8. [Supporting Modules](#8-supporting-modules)

---
## 1. Project Overview

### Directory Structure

```
AIReceptionist-main/
├── receptionist/              # Main Python package (~8,500 lines)
│   ├── agent.py               # Entry point & session handler (2,761 lines)
│   ├── config.py              # Pydantic config models (1,084 lines)
│   ├── prompts.py             # System prompt builder (334 lines)
│   ├── lifecycle.py           # Per-call state management (509 lines)
│   ├── voice_auth.py          # OpenAI Realtime auth resolver
│   ├── info_packets.py        # Info packet email delivery
│   ├── booking/               # Google Calendar integration
│   ├── email/                 # Email transport layer
│   ├── intakes/               # Structured intake forms
│   ├── messaging/             # Message delivery channels
│   ├── recording/             # Call recording (LiveKit Egress)
│   ├── transcript/            # Call transcript capture
│   └── retention/             # Data retention sweeper
├── config/businesses/         # Per-business YAML configs
├── tests/                     # ~380 unit tests
├── documentation/             # Setup guides & references
└── scripts/                   # Developer tooling
```

### Key Features

| Feature | Description |
|---------|-------------|
| FAQ Answering | Knowledge base lookup from configured Q&A pairs |
| Business Hours | Timezone-aware scheduling with open/close times |
| Call Transfers | SIP-based routing to departments/people |
| Message Taking | Multi-channel delivery (file, email, webhook) |
| Appointment Booking | Google Calendar integration with race detection |
| Intake Forms | Structured data collection with DTMF support |
| Call Recording | Local or S3 storage via LiveKit Egress |

---
## 2. Data Flow Architecture

### High-Level Call Flow

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                           INBOUND CALL FLOW                                  │
└─────────────────────────────────────────────────────────────────────────────┘

  ┌──────────┐     ┌──────────┐     ┌──────────────┐     ┌──────────────────┐
  │  Caller  │────▶│ SIP/PSTN │────▶│   LiveKit    │────▶│  AgentServer     │
  │  Phone   │     │  Trunk   │     │   Server     │     │  (agent.py)      │
  └──────────┘     └──────────┘     └──────────────┘     └────────┬─────────┘
                                                                   │
                                                                   ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│                          AGENT SESSION SETUP                                 │
├─────────────────────────────────────────────────────────────────────────────┤
│  1. Load business config (YAML → Pydantic models)                           │
│  2. Build system prompt from config                                          │
│  3. Create CallLifecycle (per-call state)                                   │
│  4. Initialize OpenAI Realtime session                                       │
│  5. Start recording if enabled                                               │
│  6. Attach transcript capture                                                │
└─────────────────────────────────────────────────────────────────────────────┘
                                                                   │
                                                                   ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│                         CONVERSATION LOOP                                    │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│   Caller speaks ──▶ OpenAI Realtime ──▶ AI Response ──▶ Caller hears        │
│                           │                                                  │
│                           ▼                                                  │
│                    ┌─────────────────┐                                       │
│                    │  Tool Calls?    │                                       │
│                    └────────┬────────┘                                       │
│                             │                                                │
│         ┌───────────┬───────┴───────┬───────────┬───────────┐               │
│         ▼           ▼               ▼           ▼           ▼               │
│   ┌──────────┐ ┌──────────┐ ┌────────────┐ ┌─────────┐ ┌─────────┐          │
│   │lookup_faq│ │transfer_ │ │take_message│ │check_   │ │end_call │          │
│   │          │ │call      │ │            │ │availab. │ │         │          │
│   └──────────┘ └──────────┘ └────────────┘ └─────────┘ └─────────┘          │
│                                                                              │
└─────────────────────────────────────────────────────────────────────────────┘
                                                                   │
                                                                   ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│                          CALL END FAN-OUT                                    │
├─────────────────────────────────────────────────────────────────────────────┤
│  1. Stop recording → get artifact URL                                        │
│  2. Write transcripts (JSON + Markdown)                                      │
│  3. Generate AI summary (if enabled)                                         │
│  4. Send emails (message, intake, call-end summary)                          │
│  5. Update metadata with outcomes                                            │
└─────────────────────────────────────────────────────────────────────────────┘
```

### Component Interaction Diagram

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                              agent.py                                        │
│  ┌─────────────────────────────────────────────────────────────────────┐    │
│  │                       handle_call()                                  │    │
│  │  Entry point - orchestrates the entire call session                  │    │
│  └─────────────────────────────────────────────────────────────────────┘    │
│                                    │                                         │
│          ┌────────────────────────┼────────────────────────┐                │
│          ▼                        ▼                        ▼                │
│  ┌──────────────┐        ┌──────────────┐        ┌──────────────┐           │
│  │  config.py   │        │  prompts.py  │        │ lifecycle.py │           │
│  │              │        │              │        │              │           │
│  │ load_config()│───────▶│build_system_ │        │CallLifecycle │           │
│  │ BusinessConf │        │   prompt()   │        │              │           │
│  └──────────────┘        └──────────────┘        └──────────────┘           │
│                                                          │                   │
│  ┌──────────────────────────────────────────────────────┘                   │
│  │                                                                           │
│  │  ┌─────────────────────────────────────────────────────────────────┐     │
│  │  │                    Receptionist (Agent)                          │     │
│  │  │  ┌───────────────────────────────────────────────────────────┐  │     │
│  │  │  │                    @function_tool()                        │  │     │
│  │  │  │  lookup_faq | transfer_call | take_message | end_call     │  │     │
│  │  │  │  check_availability | book_appointment                     │  │     │
│  │  │  │  record_intake_answer | finalize_intake | send_info_packet │  │     │
│  │  │  └───────────────────────────────────────────────────────────┘  │     │
│  │  └─────────────────────────────────────────────────────────────────┘     │
│  │                                    │                                      │
│  │              ┌─────────────────────┼─────────────────────┐               │
│  ▼              ▼                     ▼                     ▼               │
│  ┌──────────┐ ┌──────────┐ ┌──────────────┐ ┌──────────────┐                │
│  │messaging/│ │ booking/ │ │   intakes/   │ │   email/     │                │
│  │Dispatcher│ │GoogleCal │ │IntakeStorage │ │EmailSender   │                │
│  └──────────┘ └──────────┘ └──────────────┘ └──────────────┘                │
└─────────────────────────────────────────────────────────────────────────────┘
```

---
## 3. Configuration System

The configuration system is built with **Pydantic v2** for type-safe validation. All business settings come from YAML files in `config/businesses/`.

### 3.1 Configuration Loading Flow

```
YAML File (config/businesses/example-dental.yaml)
                    │
                    ▼
            yaml.safe_load()
                    │
                    ▼
         _interpolate_env_vars()  ←── ${ENV_VAR} substitution
                    │
                    ▼
         BusinessConfig.model_validate()
                    │
                    ▼
         Pydantic Validation (field validators, model validators)
                    │
                    ▼
         BusinessConfig instance (ready to use)
```

In [ ]:
# Example: Loading a business configuration
# This is the core function that loads and validates a YAML config

from pathlib import Path
import yaml

# Simulated config loading (actual code from config.py)
def load_config(path):
    """Load and validate a business configuration from YAML."""
    text = Path(path).read_text(encoding="utf-8")
    # BusinessConfig.from_yaml_string handles:
    # 1. YAML parsing with yaml.safe_load()
    # 2. Environment variable interpolation
    # 3. Pydantic model validation
    return text  # In real code: BusinessConfig.from_yaml_string(text)

# Display the example config structure
print("Example config path: config/businesses/example-dental.yaml")

### 3.2 Core Configuration Models

The `config.py` file defines Pydantic models for every configuration section:

In [ ]:
# Key Pydantic models from config.py (simplified for illustration)

from pydantic import BaseModel, Field, field_validator
from typing import Literal

class BusinessInfo(BaseModel):
    """Basic business information."""
    name: str              # "Acme Dental"
    type: str              # "dental office"
    timezone: str          # "America/New_York"
    
    @field_validator("timezone")
    @classmethod
    def validate_timezone(cls, v: str) -> str:
        """Validate timezone is a valid IANA timezone."""
        from zoneinfo import ZoneInfo, ZoneInfoNotFoundError
        try:
            ZoneInfo(v)
        except ZoneInfoNotFoundError as e:
            raise ValueError(f"Invalid IANA timezone: {v!r}") from e
        return v

class DayHours(BaseModel):
    """Business hours for a single day."""
    open: str   # "08:00" (24-hour format)
    close: str  # "17:00"
    
class WeeklyHours(BaseModel):
    """Business hours for the entire week."""
    monday: DayHours | None = None
    tuesday: DayHours | None = None
    wednesday: DayHours | None = None
    thursday: DayHours | None = None
    friday: DayHours | None = None
    saturday: DayHours | None = None  # None = closed
    sunday: DayHours | None = None

class RoutingEntry(BaseModel):
    """A department/person the call can be transferred to."""
    name: str          # "Front Desk"
    number: str        # "+15551234567"
    description: str   # "General inquiries, scheduling"

class FAQEntry(BaseModel):
    """A frequently asked question and its answer."""
    question: str  # "Do you accept my insurance?"
    answer: str    # "We accept most major dental insurance..."

print("Configuration models define the schema for business settings.")
print("Each model uses Pydantic validation to ensure data integrity.")

### 3.3 Message Channels (Discriminated Union)

The configuration uses Pydantic's discriminated unions for type-safe channel definitions:

In [ ]:
# Message channel types - uses discriminated union pattern
from typing import Union, Annotated
from pydantic import Field

class FileChannel(BaseModel):
    """Save messages to a JSON file."""
    type: Literal["file"]
    file_path: str  # "./messages/acme-dental/"

class EmailChannel(BaseModel):
    """Send messages via email."""
    type: Literal["email"]
    to: list[str]  # ["owner@acmedental.com"]
    include_transcript: bool = True
    include_recording_link: bool = True

class WebhookChannel(BaseModel):
    """Post messages to a webhook URL."""
    type: Literal["webhook"]
    url: str  # "https://hooks.slack.com/services/..."
    headers: dict[str, str] = {}

# Discriminated union - Pydantic uses the 'type' field to determine which model
MessageChannel = Annotated[
    Union[FileChannel, EmailChannel, WebhookChannel],
    Field(discriminator="type"),
]

print("Discriminated unions allow type-safe handling of different channel types.")
print("Example YAML:")
print("""
messages:
  channels:
    - type: "file"
      file_path: "./messages/acme-dental/"
    - type: "email"
      to: ["owner@acmedental.com"]
    - type: "webhook"
      url: "https://hooks.slack.com/..."
""")

### 3.4 Top-Level BusinessConfig

The `BusinessConfig` model combines all sections:

In [ ]:
# The top-level config model (simplified)

class BusinessConfig(BaseModel):
    """Complete business configuration."""
    
    # Required sections
    business: "BusinessInfo"        # Name, type, timezone
    greeting: str                    # "Thank you for calling..."
    personality: str                 # AI personality instructions
    hours: "WeeklyHours"            # Business hours
    after_hours_message: str         # What to say when closed
    routing: list["RoutingEntry"]   # Transfer destinations
    faqs: list["FAQEntry"]          # FAQ knowledge base
    messages: "MessagesConfig"       # Message delivery channels
    
    # Optional sections (enabled via YAML)
    voice: "VoiceConfig" = None     # Voice model settings
    languages: "LanguagesConfig" = None  # Multi-language support
    recording: "RecordingConfig" = None  # Call recording
    transcripts: "TranscriptsConfig" = None  # Transcript export
    email: "EmailConfig" = None      # Email sender config
    calendar: "CalendarConfig" = None  # Google Calendar
    intakes: "IntakesConfig" = None   # Structured intake forms
    sip: "SipConfig" = None          # SIP transfer settings
    dtmf: "DtmfConfig" = None        # Keypad menu
    info_packets: "InfoPacketsConfig" = None  # Info packet emails

print("BusinessConfig is the root model that validates the entire YAML file.")

### 3.5 Environment Variable Interpolation

YAML configs can reference environment variables using `${VAR_NAME}` syntax:

In [ ]:
# Environment variable interpolation from config.py
import re
import os

_ENV_PATTERN = re.compile(r"\$\{([A-Z_][A-Z0-9_]*)\}")

def _interpolate_env_vars(node):
    """Recursively replace ${VAR} placeholders with environment values."""
    if isinstance(node, str):
        def _replace(match):
            var = match.group(1)
            if var not in os.environ:
                raise ValueError(f"Environment variable {var} not set")
            return os.environ[var]
        return _ENV_PATTERN.sub(_replace, node)
    if isinstance(node, dict):
        return {k: _interpolate_env_vars(v) for k, v in node.items()}
    if isinstance(node, list):
        return [_interpolate_env_vars(v) for v in node]
    return node

# Example usage
print("YAML with env vars:")
print("""
email:
  sender:
    smtp:
      username: ${SMTP_USERNAME}
      password: ${SMTP_PASSWORD}
""")

---
## 4. System Prompt Building

The `prompts.py` module transforms the business configuration into a system prompt that instructs the OpenAI Realtime model how to behave.

### 4.1 Prompt Structure

In [ ]:
# System prompt building from prompts.py

def build_system_prompt(config):
    """Build the system prompt from business configuration."""
    
    # Build business hours block
    hours_lines = []
    for day_name in ["monday", "tuesday", "wednesday", "thursday",
                     "friday", "saturday", "sunday"]:
        day_hours = getattr(config.hours, day_name)
        if day_hours is None:
            hours_lines.append(f"  {day_name.capitalize()}: Closed")
        else:
            hours_lines.append(f"  {day_name.capitalize()}: {day_hours.open} - {day_hours.close}")
    hours_block = "\n".join(hours_lines)

    # Build routing block
    routing_lines = [f"  - {e.name}: {e.description}" for e in config.routing]
    routing_block = "\n".join(routing_lines)

    # Build FAQ block
    faq_lines = [f"  Q: {faq.question}\n  A: {faq.answer}" for faq in config.faqs]
    faq_block = "\n\n".join(faq_lines)

    # Assemble the full prompt
    return f"""You are the receptionist for {config.business.name}, a {config.business.type}.

{config.personality}

BUSINESS HOURS (timezone: {config.business.timezone}):
{hours_block}

When the business is closed, say: {config.after_hours_message}

DEPARTMENTS YOU CAN TRANSFER TO:
{routing_block}

FREQUENTLY ASKED QUESTIONS:
{faq_block}

IMPORTANT RULES:
- Be concise. Phone conversations should be efficient.
- Never make up information. If you don't know, say so and offer alternatives.
- Always confirm before transferring a call.
- If the caller seems upset, be empathetic and offer to connect them with a person.
"""

print("The system prompt is dynamically built from the YAML configuration.")
print("It includes:")
print("  - Business identity and personality")
print("  - Business hours in the local timezone")
print("  - Transfer destinations with descriptions")
print("  - FAQ knowledge base")
print("  - Behavioral rules and constraints")

### 4.2 Conditional Prompt Sections

The prompt builder adds sections based on enabled features:

In [ ]:
# Conditional sections added to the prompt

def _build_calendar_block(config):
    """Build CALENDAR section if enabled."""
    if config.calendar is None or not config.calendar.enabled:
        return ""
    return """
CALENDAR (appointment booking):
You can book appointments on the business calendar using two tools:
  1. check_availability(preferred_date, preferred_time) — call this FIRST.
     It returns up to 3 available slots near the caller's preferred time.
  2. book_appointment(caller_name, callback_number, proposed_start_iso,
     notes, caller_email) — call this AFTER the caller confirms the time.

BOOKING CONVENTIONS:
  - Before booking, always say the specific time back to the caller and wait
    for explicit confirmation.
  - Always read back the callback NUMBER digit-by-digit.
  - NEVER fabricate a time, confirmation code, or event ID.
"""

def _build_intakes_block(config):
    """Build INTAKES section if enabled."""
    if config.intakes is None or not config.intakes.enabled:
        return ""
    # Lists all case types and their questions
    # Includes detailed procedure instructions
    return "[Intakes section with case types and procedures...]"

def _build_language_block(config):
    """Build LANGUAGE section for multi-language support."""
    primary = config.languages.primary
    allowed = config.languages.allowed
    
    if len(allowed) <= 1:
        return f"""LANGUAGE:
Speak {primary} only. If the caller speaks another language,
politely say you can only assist in {primary}."""
    else:
        return f"""LANGUAGE:
Your primary language is {primary}. You can also respond in: {', '.join(allowed)}.
If the caller speaks one of those languages, respond in that language."""

print("Prompt sections are conditionally included based on YAML configuration:")
print("  - CALENDAR section: only if calendar.enabled = true")
print("  - INTAKES section: only if intakes.enabled = true")
print("  - INFO PACKETS section: only if info_packets.enabled = true")
print("  - DTMF MENU section: only if dtmf.enabled = true")

---
## 5. Agent Entry Point

The `agent.py` file contains the main entry point (`handle_call`) and the `Receptionist` class that implements all the AI tools.

### 5.1 Agent Server Setup

In [ ]:
# Agent server setup (simplified from agent.py)
# This is how the agent is initialized and started

# from livekit.agents import AgentServer, AgentSession, Agent, function_tool

# The agent server is created with an entrypoint function
# server = AgentServer(
#     request_fnc=handle_call,  # Called for each incoming call
#     setup_fnc=_prewarm,       # Called once per worker subprocess
#     agent_name=_resolve_agent_name(),
# )

# To run: python -m receptionist.agent dev

print("The AgentServer:")
print("  1. Listens for incoming SIP calls via LiveKit")
print("  2. Spawns a handle_call() coroutine for each call")
print("  3. Manages worker processes for scalability")

### 5.2 The handle_call Function

This is the main entry point for each incoming call:

In [ ]:
# Simplified handle_call from agent.py

async def handle_call(ctx):
    """Entry point for each incoming call."""
    
    # 1. Load business configuration
    config = load_business_config(ctx)
    
    # 2. Extract caller information
    caller_phone = _get_caller_phone(ctx)
    call_id = ctx.room.name  # Unique room name = call ID
    
    # 3. Create CallLifecycle (per-call state manager)
    lifecycle = CallLifecycle(
        config=config,
        call_id=call_id,
        caller_phone=caller_phone,
    )
    
    # 4. Create the Receptionist agent with tools
    receptionist = Receptionist(config, lifecycle)
    
    # 5. Initialize OpenAI Realtime session
    api_key = await resolve_voice_bearer_async(config.voice.auth)
    realtime_model = openai.realtime.RealtimeModel(
        model=config.voice.model,
        voice=config.voice.voice_id,
        api_key=api_key,
    )
    
    # 6. Create and start the agent session
    session = AgentSession(
        realtime_model=realtime_model,
        agent=receptionist,
    )
    
    # 7. Start recording if enabled
    await lifecycle.start_recording_if_enabled(ctx.room.name)
    
    # 8. Attach transcript capture
    lifecycle.attach_transcript_capture(session)
    
    # 9. Set up event handlers (silence timeout, DTMF, etc.)
    # ...
    
    # 10. Start the session and begin conversation
    await session.start()
    
    # The session runs until the call ends
    # on_call_ended() is called during cleanup

print("handle_call() orchestrates the entire call lifecycle:")
print("  - Config loading and validation")
print("  - Caller identification")
print("  - Agent and session initialization")
print("  - Recording and transcript setup")
print("  - Event handler registration")

### 5.3 The Receptionist Class

The `Receptionist` class extends `Agent` and implements all the function tools:

In [ ]:
# Receptionist class structure (simplified from agent.py)

class Receptionist:
    """The AI receptionist agent with all function tools."""
    
    def __init__(self, config, lifecycle):
        # Initialize with system prompt from config
        super().__init__(instructions=build_system_prompt(config))
        
        self.config = config
        self.lifecycle = lifecycle
        
        # Session state
        self._offered_slot_batches = []  # Slots offered to caller
        self._calendar_client = None     # Lazy-loaded Google Calendar
        self._intake_answers = {}        # Collected intake answers
        self._intake_case_type = None    # Current intake type
        
        # Pre-build dispatcher for message delivery
        self._dispatcher = Dispatcher(
            channels=config.messages.channels,
            business_name=config.business.name,
        )
        
        # Build routing lookup table
        self._routing_by_name = {
            r.name.lower(): r for r in config.routing
        }
    
    async def on_enter(self):
        """Called when the agent session starts."""
        # Speak consent preamble if recording
        if self.config.recording and self.config.recording.enabled:
            if self.config.recording.consent_preamble.enabled:
                await self.session.generate_reply(
                    instructions="Say: 'This call may be recorded...'"
                )
        
        # Speak the greeting
        await self.session.generate_reply(
            instructions=f"Greet: {self.config.greeting}"
        )

print("The Receptionist class:")
print("  - Inherits from Agent base class")
print("  - Receives system prompt from build_system_prompt()")
print("  - Manages per-call state (slots, intakes, calendar)")
print("  - Implements all @function_tool() methods")

---
## 6. Call Lifecycle Management

The `lifecycle.py` module manages per-call state and handles the end-of-call cleanup.

### 6.1 CallLifecycle Class

In [ ]:
# CallLifecycle from lifecycle.py (simplified)

class CallLifecycle:
    """Owns per-call state and the disconnect-time fan-out."""
    
    def __init__(self, config, call_id, caller_phone):
        self.config = config
        self.metadata = CallMetadata(
            call_id=call_id,
            business_name=config.business.name,
            caller_phone=caller_phone,
        )
        
        self.transcript_capture = None
        self.recording_handle = None
        
        # Deferred email delivery (sent at call end)
        self._pending_message_emails = []
        self._pending_intake_submission = None
        
        # Guard against double-firing
        self._finalized = False
    
    # --- Outcome Recorders ---
    
    def record_transfer(self, department_name):
        """Record that a transfer was made."""
        self.metadata.transfer_target = department_name
        self.metadata.outcomes.add("transferred")
    
    def record_message_taken(self):
        """Record that a message was taken."""
        self.metadata.message_taken = True
        self.metadata.outcomes.add("message_taken")
    
    def record_appointment_booked(self, details):
        """Record that an appointment was booked."""
        self.metadata.appointment_booked = True
        self.metadata.appointment_details = details
        self.metadata.outcomes.add("appointment_booked")
    
    def enqueue_message_email(self, message):
        """Queue a message for email at call end."""
        self._pending_message_emails.append(message)

print("CallLifecycle tracks:")
print("  - Call metadata (ID, caller phone, timestamps)")
print("  - Outcomes (transferred, message_taken, appointment_booked)")
print("  - Recording and transcript handles")
print("  - Pending emails to send at call end")

### 6.2 End-of-Call Fan-Out

When the call ends, `on_call_ended()` performs cleanup and notifications:

In [ ]:
# on_call_ended from lifecycle.py (simplified)

async def on_call_ended(self):
    """Called when the call disconnects."""
    
    # Guard: only run once
    if self._finalized:
        return
    self._finalized = True
    self.metadata.mark_finalized()
    
    # 1. Stop recording and get artifact URL
    artifact = None
    if self.recording_handle:
        artifact = await stop_recording(self.recording_handle)
        self.metadata.recording_artifact = artifact.url
    
    # 2. Write transcript files (JSON + Markdown)
    segments = self.transcript_capture.segments if self.transcript_capture else []
    transcript_result = await write_transcript_files(
        self.config.transcripts, self.metadata, segments
    )
    
    # 3. Generate AI summary (if enabled)
    ai_summary = None
    if self.config.email and self.config.email.summary.enabled:
        ai_summary = await generate_call_summary(
            segments=segments,
            metadata=self.metadata,
        )
    
    # 4. Send emails (consolidated or separate triggers)
    if self.config.email:
        if self.config.email.triggers.on_call_end:
            # One consolidated email with everything
            await self._fire_email_trigger("call_end", ...)
        else:
            # Separate emails per trigger
            if self._pending_message_emails:
                await self._send_message_emails()
            if self._pending_intake_submission:
                await self._send_intake_email()

print("on_call_ended() sequence:")
print("  1. Stop recording → get recording URL")
print("  2. Write transcripts (JSON + Markdown files)")
print("  3. Generate AI summary of the call")
print("  4. Send emails (message, intake, call-end summary)")

---
## 7. Tool Implementations

The Receptionist class implements several function tools that the AI can invoke during calls.

### 7.1 Core Tools Overview

In [ ]:
# Tool implementations from agent.py

tools_overview = """
┌─────────────────────────────────────────────────────────────────────────────┐
│                         RECEPTIONIST TOOLS                                   │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│  INFORMATION TOOLS                                                           │
│  ├── lookup_faq(question)          Search FAQ knowledge base                │
│  └── get_business_hours()          Check if currently open/closed           │
│                                                                              │
│  COMMUNICATION TOOLS                                                         │
│  ├── transfer_call(department)     SIP transfer to department/person        │
│  ├── take_message(name, msg, #)    Save message to channels                 │
│  └── end_call(reason)              Gracefully hang up                       │
│                                                                              │
│  CALENDAR TOOLS (if enabled)                                                 │
│  ├── check_availability(date, time) Find available appointment slots        │
│  └── book_appointment(name, #, iso) Book a calendar event                   │
│                                                                              │
│  INTAKE TOOLS (if enabled)                                                   │
│  ├── record_intake_answer(...)     Store one intake question answer         │
│  ├── finalize_intake(...)          Complete the intake submission           │
│  └── await_keypad_entry(key)       Capture DTMF digits for a field          │
│                                                                              │
│  INFO PACKET TOOLS (if enabled)                                              │
│  └── send_info_packet(key, email)  Email pre-configured info packet         │
│                                                                              │
└─────────────────────────────────────────────────────────────────────────────┘
"""
print(tools_overview)

### 7.2 lookup_faq Tool

In [ ]:
# lookup_faq implementation

# @function_tool()
async def lookup_faq(self, ctx, question: str) -> str:
    """Look up the answer to a frequently asked question."""
    
    # Bidirectional substring matching:
    # - Caller "hours" matches FAQ "What are your hours?"
    # - FAQ "insurance" matches caller question about insurance
    for faq in self.config.faqs:
        if (question.lower() in faq.question.lower() or 
            faq.question.lower() in question.lower()):
            # Record for transcript/analytics
            self.lifecycle.record_faq_answered(faq.question)
            return faq.answer
    
    return "No exact FAQ match found. Use your knowledge from the system prompt."

print("lookup_faq:")
print("  - Input: caller's question")
print("  - Matching: bidirectional substring search")
print("  - Output: FAQ answer or fallback message")

### 7.3 transfer_call Tool

In [ ]:
# transfer_call implementation

# @function_tool()
async def transfer_call(self, ctx, department: str) -> str:
    """Transfer the caller to a specific department or person."""
    result = await self._execute_transfer(department)
    return result.message

async def _execute_transfer(self, department: str):
    """Shared transfer logic for tool and DTMF paths."""
    
    # Check intake_only mode
    if self.config.agent.mode == "intake_only":
        return TransferResult(
            status="intake_only_refused",
            message="This line cannot transfer calls."
        )
    
    # Look up department (case-insensitive)
    target = self._routing_by_name.get(department.lower())
    if target is None:
        available = ", ".join(e.name for e in self.config.routing)
        return TransferResult(
            status="department_not_found",
            message=f"Department '{department}' not found. Available: {available}"
        )
    
    # Execute SIP transfer via LiveKit API
    job_ctx = get_job_context()
    await job_ctx.api.sip.transfer_sip_participant(
        api.TransferSIPParticipantRequest(
            room_name=job_ctx.room.name,
            participant_identity=_get_caller_identity(job_ctx),
            transfer_to=self.config.sip.transfer_uri_template.format(
                number=target.number
            ),
        )
    )
    
    self.lifecycle.record_transfer(target.name)
    return TransferResult(
        status="transferred",
        message=f"Call transferred to {target.name}"
    )

print("transfer_call:")
print("  1. Check if transfers are allowed (intake_only mode)")
print("  2. Look up department in routing table")
print("  3. Execute SIP transfer via LiveKit API")
print("  4. Record outcome in lifecycle")

### 7.4 take_message Tool

In [ ]:
# take_message implementation

# @function_tool()
async def take_message(
    self, ctx,
    caller_name: str,
    message: str,
    callback_number: str
) -> str:
    """Take a message from the caller."""
    
    # Apply input caps (truncate overlong fields)
    caller_name = _cap("caller_name", caller_name)  # max 200 chars
    message = _cap("message", message)              # max 4000 chars
    callback_number = _cap("callback_number", callback_number)  # max 50 chars
    
    # Create message object
    msg = Message(
        caller_name=caller_name,
        callback_number=callback_number,
        message=message,
        business_name=self.config.business.name,
    )
    
    # Dispatch to file/webhook channels immediately
    # (Email is deferred to call-end for transcript attachment)
    await self._dispatcher.dispatch_message(
        msg,
        context,
        skip_email_channel=True,  # Email sent at call end
    )
    
    # Queue email for later
    self.lifecycle.enqueue_message_email(msg)
    self.lifecycle.record_message_taken()
    
    return f"Message saved. Let them know someone will call back."

print("take_message:")
print("  1. Truncate overlong fields (security/storage)")
print("  2. Create Message dataclass")
print("  3. Dispatch to file/webhook immediately")
print("  4. Queue email for call-end (with transcript)")
print("  5. Record outcome in lifecycle")

### 7.5 Calendar Tools (check_availability, book_appointment)

In [ ]:
# Calendar tools implementation

# @function_tool()
async def check_availability(self, ctx, preferred_date: str, preferred_time: str) -> str:
    """Check calendar availability for appointments."""
    
    # Get or create calendar client
    client = await self._get_calendar_client_async()
    
    # Parse and resolve relative dates ("tomorrow", "next Monday")
    now = datetime.now(ZoneInfo(self.config.business.timezone))
    resolved_date = _resolve_relative_date(preferred_date, now)
    
    # Find available slots
    slots = await find_slots(
        client=client,
        config=self.config.calendar,
        preferred_date=resolved_date,
        preferred_time=preferred_time,
        business_hours=self.config.hours,
        timezone=self.config.business.timezone,
    )
    
    # Cache offered slots (prevents LLM from hallucinating times)
    self._record_offered_slots([s.iso for s in slots])
    
    # Format for speech
    return format_slots_for_speech(slots)

# @function_tool()
async def book_appointment(self, ctx, caller_name: str, callback_number: str,
                           proposed_start_iso: str, notes: str = "",
                           caller_email: str = None) -> str:
    """Book an appointment on the calendar."""
    
    # CRITICAL: Validate the ISO was actually offered
    if not self._slot_was_offered(proposed_start_iso):
        return ("That time wasn't offered. Call check_availability first "
                "and use one of the returned times.")
    
    # Book with race detection
    result = await book_appointment_with_race_check(
        client=self._calendar_client,
        config=self.config.calendar,
        proposed_start_iso=proposed_start_iso,
        caller_name=caller_name,
        callback_number=callback_number,
        notes=notes,
        caller_email=caller_email,
    )
    
    self.lifecycle.record_appointment_booked(result.details)
    return f"Booked: {result.friendly_time}"

print("Calendar tools:")
print("  check_availability:")
print("    - Resolves relative dates ('tomorrow', 'next Monday')")
print("    - Queries Google Calendar for free/busy")
print("    - Returns up to 3 available slots")
print("    - Caches offered ISOs (prevents hallucination)")
print("  book_appointment:")
print("    - Validates ISO was actually offered")
print("    - Re-checks availability (race detection)")
print("    - Creates calendar event")
print("    - Optionally sends .ics invite")

---
## 8. Supporting Modules

### 8.1 Messaging System (messaging/)

In [ ]:
# Messaging architecture

messaging_diagram = """
┌─────────────────────────────────────────────────────────────────────────────┐
│                          MESSAGING SYSTEM                                    │
└─────────────────────────────────────────────────────────────────────────────┘

                           ┌──────────────┐
                           │  Dispatcher  │
                           │              │
                           │ dispatch_    │
                           │ message()    │
                           └──────┬───────┘
                                  │
           ┌──────────────────────┼──────────────────────┐
           │                      │                      │
           ▼                      ▼                      ▼
    ┌─────────────┐        ┌─────────────┐        ┌─────────────┐
    │ FileChannel │        │WebhookChan. │        │EmailChannel │
    │             │        │             │        │             │
    │ ./messages/ │        │ HTTP POST   │        │ SMTP/Resend │
    │ JSON files  │        │ with retry  │        │ async       │
    └─────────────┘        └─────────────┘        └─────────────┘
           │                      │                      │
           │ SYNC                 │ BACKGROUND           │ DEFERRED
           │ (awaited)            │ (retry policy)       │ (call-end)
           ▼                      ▼                      ▼
    ┌─────────────┐        ┌─────────────┐        ┌─────────────┐
    │  messages/  │        │   Slack/    │        │   Email     │
    │  <business>/│        │  External   │        │  with       │
    │  <uuid>.json│        │   Service   │        │ transcript  │
    └─────────────┘        └─────────────┘        └─────────────┘

DELIVERY PRIORITY:
  1. FileChannel (sync) - always awaited first, most reliable
  2. WebhookChannel (background) - 3 retries with exponential backoff
  3. EmailChannel (deferred) - sent at call-end with transcript attachment
"""
print(messaging_diagram)

### 8.2 Email System (email/)

In [ ]:
# Email system architecture

email_diagram = """
┌─────────────────────────────────────────────────────────────────────────────┐
│                            EMAIL SYSTEM                                      │
└─────────────────────────────────────────────────────────────────────────────┘

  EMAIL SENDERS (Protocol-based):
  ┌─────────────────┐         ┌─────────────────┐
  │   SMTPSender    │         │  ResendSender   │
  │                 │         │                 │
  │  aiosmtplib     │         │  httpx POST     │
  │  async SMTP     │         │  Resend API     │
  └─────────────────┘         └─────────────────┘

  EMAIL TEMPLATES (templates.py):
  ┌───────────────────────────────────────────────────────────────┐
  │  message_received    │  New voicemail notification            │
  │  intake_submitted    │  Completed intake form                 │
  │  call_end_summary    │  Consolidated call summary             │
  │  booking_confirmation│  Appointment confirmation              │
  │  info_packet         │  Pre-approved information packet       │
  └───────────────────────────────────────────────────────────────┘

  AI SUMMARIZER (summarizer.py):
  ┌───────────────────────────────────────────────────────────────┐
  │  generate_call_summary()                                      │
  │    - Takes transcript segments + metadata                     │
  │    - Calls OpenAI chat completion (gpt-4o-mini)               │
  │    - Returns 2-4 sentence summary                             │
  │    - 20-second timeout, graceful degradation                  │
  └───────────────────────────────────────────────────────────────┘
"""
print(email_diagram)

### 8.3 Booking System (booking/)

In [ ]:
# Booking system architecture

booking_diagram = """
┌─────────────────────────────────────────────────────────────────────────────┐
│                          BOOKING SYSTEM                                      │
└─────────────────────────────────────────────────────────────────────────────┘

                        ┌──────────────────┐
                        │ GoogleCalendar   │
                        │    Client        │
                        │                  │
                        │ - list_events()  │
                        │ - insert_event() │
                        │ - freebusy()     │
                        └────────┬─────────┘
                                 │
               ┌─────────────────┴─────────────────┐
               │                                   │
               ▼                                   ▼
    ┌──────────────────┐              ┌──────────────────┐
    │   find_slots()   │              │ book_appointment │
    │  (availability)  │              │    (booking)     │
    │                  │              │                  │
    │ 1. Get biz hours │              │ 1. Re-check slot │
    │ 2. Query freebusy│              │ 2. Race detect   │
    │ 3. Find gaps     │              │ 3. Insert event  │
    │ 4. Return slots  │              │ 4. Tag UNVERIFIED│
    └──────────────────┘              └──────────────────┘

  AUTH METHODS:
  ┌────────────────────┐    ┌────────────────────┐
  │  Service Account   │    │      OAuth         │
  │                    │    │                    │
  │  JSON key file     │    │  Interactive flow  │
  │  No user consent   │    │  User consent once │
  │  Calendar shared   │    │  Token refresh     │
  └────────────────────┘    └────────────────────┘
"""
print(booking_diagram)

### 8.4 Transcript System (transcript/)

In [ ]:
# Transcript system architecture

transcript_diagram = """
┌─────────────────────────────────────────────────────────────────────────────┐
│                         TRANSCRIPT SYSTEM                                    │
└─────────────────────────────────────────────────────────────────────────────┘

  CAPTURE (capture.py):
  ┌───────────────────────────────────────────────────────────────┐
  │  TranscriptCapture                                            │
  │    - Listens to AgentSession events                           │
  │    - Captures: user speech, agent responses, tool calls       │
  │    - Stores as TranscriptSegment objects                      │
  └───────────────────────────────────────────────────────────────┘
                                 │
                                 ▼
  METADATA (metadata.py):
  ┌───────────────────────────────────────────────────────────────┐
  │  CallMetadata                                                 │
  │    - call_id, caller_phone, business_name                     │
  │    - start_time, end_time                                     │
  │    - outcomes: {transferred, message_taken, ...}              │
  │    - transfer_target, appointment_details                     │
  │    - faqs_answered[], dtmf_events[]                           │
  │    - recording_artifact (URL)                                 │
  └───────────────────────────────────────────────────────────────┘
                                 │
               ┌─────────────────┴─────────────────┐
               ▼                                   ▼
  FORMATTER (formatter.py):          WRITER (writer.py):
  ┌─────────────────────┐            ┌─────────────────────┐
  │  to_json()          │            │ write_transcript_   │
  │  to_markdown()      │            │ files()             │
  │                     │            │                     │
  │  Structured JSON    │            │ Atomic file writes  │
  │  Readable Markdown  │            │ ./transcripts/      │
  └─────────────────────┘            └─────────────────────┘
"""
print(transcript_diagram)

---
## Summary

### Key Takeaways

1. **Configuration-Driven**: All business behavior comes from YAML files validated by Pydantic

2. **Async-First**: All I/O is wrapped in `asyncio.to_thread()` to prevent event loop blocking

3. **Deterministic Messaging**: File channel always syncs first; others run in background with retry

4. **Security by Default**: Path validation, safe YAML loading, environment variable templating

5. **Multi-Business Capable**: Single agent process handles multiple configs via SIP metadata

6. **Testability**: Subpackages have mockable surfaces; ~380 unit tests

7. **Error Resilience**: Graceful degradation (DTMF fallback, retry with backoff, no-op on send failures)

8. **Observability**: Structured logging, transcript capture, email audit trail

### Next Steps

To explore further:
1. Read `config/businesses/example-dental.yaml` for a complete config example
2. Review `tests/` for unit test patterns
3. Check `documentation/` for deployment and setup guides